# 55 — Shadow Deployment & Rollback Drill

Valida la máquina de estados de despliegue y el mecanismo de pausa/rollback sin mutar destructivamente el estado real. `production` no existe como transición habilitada en M6 V1.

In [1]:
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


{'m6_schema': 'm6-production-v1',
 'model_version': 'argos-v1',
 'operating_mode': 'shadow',
 'scoring_mode': 'batch',
 'human_review_required': True,
 'production_promotion_authorized': False}

In [2]:
from argos.production.deployment import BUILD_IMPLEMENTATION, run_shadow_rollback_drill

print("implementation:", BUILD_IMPLEMENTATION)
result = run_shadow_rollback_drill(
    root=paths.root, production_spec=M6_SPEC, reports_root=paths.reports, manifest_dir=MANIFEST_DIR
)
result.drill

implementation: m6-55-shadow-rollback-drill-v1


,from_stage,to_stage,reason,allowed
0,candidate,shadow,shadow promotion,True
1,shadow,paused,synthetic high-severity alert,True
2,paused,candidate,rollback to known safe candidate,True
3,candidate,production,forbidden autonomous promotion path,False


In [3]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,shadow_pause_rollback_transitions,rollback,True,True,"[{""allowed"": true, ""from_stage"": ""candidate"", ...",candidate→shadow→paused→candidate rollback pat...
1,production_transition_not_available,governance,True,True,"{""config_allow_production"": false, ""production...",M6 V1 has no accidental direct path to autonom...
2,audit_hash_chain_valid,audit,True,True,"{""error"": null, ""events"": 15}",operational audit evidence is append-only and ...
3,rollback_drill_is_non_destructive,operations,True,True,"{""deployment_state_not_mutated"": true, ""mode"":...",rollback mechanism is tested without mutating ...


In [4]:
blocking = result.validation.loc[result.validation["blocking"]]
print("55 ✅ PASS" if result.ready_for_next_notebook else "55 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook

55 ✅ PASS
Blocking checks: 4/4
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m6_shadow_rollback.json
